# AH | Enriquecimiento territorial

**Autor:** Alan Arturo Hernández Gutiérrez.

Une los catálogos por clave municipal y conserva los indicadores del total municipal de INEGI. Los valores ausentes mantienen su condición de dato faltante.

Este cuaderno define las operaciones que utilizan los cuadernos de `notebooks/`. Ejecute sus celdas en orden. El flujo completo se inicia desde 01 o 02.

## Raíz del proyecto

La ubicación se resuelve desde la raíz, `src/` o `notebooks/`, sin guardar rutas personales.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name.lower() in {'src', 'notebooks'}:
    ROOT = ROOT.parent
if not (ROOT / 'config' / 'proyecto.json').is_file():
    raise RuntimeError('Inicie Jupyter desde la raíz del proyecto, src/ o notebooks/.')


## Bibliotecas y parámetros

Se preparan las bibliotecas y constantes utilizadas por las operaciones siguientes.

In [ ]:
from pathlib import Path

import hashlib

import json

import zipfile

import pandas as pd


## Identificar la versión del archivo

Calcula SHA-256 por bloques. El resultado permite comprobar la versión de una entrada o una salida sin cargar todo el archivo en memoria.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


## Cargar iter municipal

Selecciona únicamente totales municipales y calcula la proporción de viviendas con Internet.

In [ ]:
def cargar_iter_municipal(path):
    """Read ONLY total-municipality records; do not sum localities and totals."""
    path = Path(path)
    required = ["ENTIDAD", "NOM_ENT", "MUN", "NOM_MUN", "LOC", "POBTOT", "VIVPARH_CV", "VPH_INTER"]
    parts = []
    with zipfile.ZipFile(path) as z:
        members = [
            n
            for n in z.namelist()
            if "conjunto_de_datos" in n.lower() and n.lower().endswith(".csv")
        ]
        if len(members) != 1:
            raise ValueError("ITER: se esperaba un CSV de conjunto de datos nacional o estatal")
        with z.open(members[0]) as f:
            for chunk in pd.read_csv(
                f, dtype="string", usecols=required, chunksize=20000, encoding="utf-8-sig"
            ):
                for c in required:
                    chunk[c] = chunk[c].str.strip()
                ent = chunk.ENTIDAD.str.zfill(2)
                mun = chunk.MUN.str.zfill(3)
                loc = chunk.LOC.str.zfill(4)
                keep = ent.ne("00") & mun.ne("000") & loc.eq("0000")
                x = chunk.loc[keep].copy()
                if x.empty:
                    continue
                x["MunicipioCve"] = ent[keep] + mun[keep]
                parts.append(x)
    if not parts:
        raise ValueError("ITER: no se encontraron totales municipales")
    d = pd.concat(parts, ignore_index=True)
    if d.MunicipioCve.duplicated().any():
        raise ValueError("ITER: claves municipales repetidas; revisar archivo y nivel geografico")
    for old, new in [
        ("POBTOT", "PoblacionMunicipio2020"),
        ("VIVPARH_CV", "ViviendasConCaracteristicas2020"),
        ("VPH_INTER", "ViviendasInternet2020"),
    ]:
        d[new] = pd.to_numeric(d[old], errors="coerce").astype("Int64")
        if d[new].dropna().lt(0).any():
            raise ValueError("ITER: indicador negativo inesperado")
    num, den = (d.ViviendasInternet2020, d.ViviendasConCaracteristicas2020)
    if (num.notna() & den.notna() & num.gt(den)).any():
        raise ValueError("ITER: viviendas con Internet superiores al denominador")
    d["PorcentajeInternetMunicipio2020"] = (num / den.where(den.gt(0))).astype("Float64")
    d["CensoMunicipalDisponible"] = 1
    cols = [
        "MunicipioCve",
        "PoblacionMunicipio2020",
        "ViviendasConCaracteristicas2020",
        "ViviendasInternet2020",
        "PorcentajeInternetMunicipio2020",
        "CensoMunicipalDisponible",
    ]
    return (
        d[cols],
        {
            "estado": "Integrado desde copia local",
            "archivo_sha256": sha256_file(path),
            "registros_municipales": len(d),
            "fuente": "INEGI Censo 2020 ITER",
            "criterio": "ENTIDAD != 00, MUN != 000, LOC = 0000",
        },
    )


## Preparar municipios

Verifica claves estatales y municipales y agrega el contexto censal cuando se proporciona el ITER.

In [ ]:
def preparar_municipios(source, estados_path, iter_path=None):
    estados = pd.read_csv(estados_path, dtype="string", encoding="utf-8-sig")
    if estados.EstadoClave.duplicated().any() or len(estados) != 32:
        raise ValueError("Catalogo estatal invalido")
    d = source.rename(
        columns={
            "MUNICIPIO_CVE": "MunicipioCve",
            "NOM_MUN": "MunicipioNombre",
            "ESTADO_CLAVE": "EstadoClave",
            "IM_2020": "IndiceTerritorialFuente2020",
        }
    ).copy()
    if d.MunicipioCve.duplicated().any() or not d.MunicipioCve.str.fullmatch("\\d{5}").all():
        raise ValueError("Catalogo municipal fuente invalido")
    if not d.MunicipioCve.str[:2].eq(d.EstadoClave).all():
        raise ValueError("Prefijo municipal/estatal incompatible")
    d = d.merge(estados, on="EstadoClave", how="left", validate="many_to_one")
    if d.NombreEstado.isna().any():
        raise ValueError("Estado sin nombre en catalogo verificado")
    metadata = {
        "estados": {
            "url": "https://gaia.inegi.org.mx/wscatgeo/v2/mgee",
            "consulta": "2026-11-05",
            "archivo_sha256": sha256_file(estados_path),
            "cobertura_municipios_fuente": int(d.NombreEstado.notna().sum()),
        }
    }
    if iter_path:
        ext, status = cargar_iter_municipal(iter_path)
        expected = json.loads((ROOT / "catalogos" / "FUENTES.json").read_text(encoding="utf-8"))[
            "iter_municipal"
        ]["archivo_sha256"]
        if status["archivo_sha256"] != expected:
            raise ValueError(
                "ITER: SHA-256 distinto de la version documentada; revisar fuente y esquema"
            )
        d = d.merge(ext, on="MunicipioCve", how="left", validate="one_to_one")
        d["CensoMunicipalDisponible"] = d.CensoMunicipalDisponible.fillna(0).astype("int8")
        status["matches_catalogo_fuente"] = int(d.CensoMunicipalDisponible.sum())
    else:
        for c in [
            "PoblacionMunicipio2020",
            "ViviendasConCaracteristicas2020",
            "ViviendasInternet2020",
        ]:
            d[c] = pd.Series(pd.NA, index=d.index, dtype="Int64")
        d["PorcentajeInternetMunicipio2020"] = pd.Series(pd.NA, index=d.index, dtype="Float64")
        d["CensoMunicipalDisponible"] = 0
        status = {
            "estado": "No integrado: falta archivo ITER local",
            "registros_municipales": 0,
            "matches_catalogo_fuente": 0,
            "no_imputado": True,
        }
    metadata["iter"] = status
    return (d, metadata)
